In [1]:
# SPDX-FileCopyrightText: 2024-2026 Dan J. Bower <dbower@eaps.ethz.ch>
# SPDX-FileCopyrightText: 2024-2026 Paolo A. Sossi
#
# SPDX-License-Identifier: Apache-2.0

from pathlib import Path
from typing import Literal

import numpy as np

from atmodeller import (
    ChemicalSpecies,
    Database,
    EquilibriumModel,
    IronWustiteBuffer,
    Parameters,
    Planet,
    PurePhase,
    ReservoirSpecies,
    console_logger,
    earth,
    get_solubility_models,
)

# Log to the console. For more detail, use console_logger(logging.DEBUG)
logger = console_logger()

# Basic usage

This notebook is available at `notebooks/examples/basic_usage.ipynb` and is easiest to obtain by downloading the source code.

We will walk through the process of defining species, setting up a model, running a basic equilibrium calculation, and accessing the output.

The scenario considered here is an Earth-sized planet with a molten silicate mantle in equilibrium with its atmosphere. Some atmospheric species have solubility relations that regulate how much of their inventory dissolves. The objective is to determine how the volatile budget partitions between the molten interior (magma ocean) and the atmosphere, and what the associated atmospheric speciation is.

## Create species

### Gas

Define the gas phase species to include in the interior-atmosphere model.

In [2]:
# Gas species
H2O_g = ChemicalSpecies.create_gas("H2O")
H2_g = ChemicalSpecies.create_gas("H2")
O2_g = ChemicalSpecies.create_gas("O2")
CO_g = ChemicalSpecies.create_gas("CO")
CO2_g = ChemicalSpecies.create_gas("CO2")
CH4_g = ChemicalSpecies.create_gas("CH4")

gas_species = (H2O_g, H2_g, O2_g, CO_g, CO2_g, CH4_g)

### Dissolved

Create dissolved species, i.e. species that are dissolved in the molten interior according to solubility relations.

In [3]:
# Various solubility models are available in Atmodeller and can be accessed via the
# `get_solubility_models` function
solubility_models = get_solubility_models()

# Set to True to include the mass of dissolved species in the phase (melt) mass. When this is set
# to False it is assumed that the mass of dissolved species is negligible compared to the mass of
# the melt, i.e. the dilute approximation is applied.
include_in_phase_mass = False

# Melt species
H2O_d: ReservoirSpecies = ReservoirSpecies.create_dissolved(
    "H2O",
    solubility=solubility_models["H2O_peridotite_sossi23"],
    include_in_phase_mass=include_in_phase_mass,
)
CO2_d: ReservoirSpecies = ReservoirSpecies.create_dissolved(
    "CO2",
    solubility=solubility_models["CO2_basalt_dixon95"],
    include_in_phase_mass=include_in_phase_mass,
)

melt_species = (H2O_d, CO2_d)

## Create model

Create a model by first defining a thermodynamic state, such as a molten Earth.

In [4]:
planet = Planet.from_species(
    gas_species,
    silicate_melt_species=melt_species,
    temperature=2000,
    mantle_melt_fraction=1.0,
    planet_mass=earth.mass,
    core_mass_fraction=earth.core_mass_fraction,
    surface_radius=earth.radius,
)

[07:41:50 - atmodeller.phases              - INFO     ] - Creating GasPhase: ('H2O_g: IdealGas', 'H2_g: IdealGas', 'O2_g: IdealGas', 'CO_g: IdealGas', 'CO2_g: IdealGas', 'CH4_g: IdealGas')


[07:41:50 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ('H2O_d: SolubilityPowerLaw', 'CO2_d: _CO2_basalt_dixon95')


[07:41:50 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[07:41:50 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[07:41:50 - atmodeller.reactions           - INFO     ] - ReactionNetwork network = {0: '1.0 H2_g + 0.5 O2_g = 1.0 H2O_g',
 1: '0.5 O2_g + 1.0 CO_g = 1.0 CO2_g',
 2: '2.0 H2_g + 1.0 CO_g = 0.5 O2_g + 1.0 CH4_g'}


[07:41:50 - atmodeller.reactions           - INFO     ] - Thermodynamic data of all species cover 200 K to 6000 K, outside which each species follows its extrapolation


[07:41:50 - atmodeller.reactions           - INFO     ] - DissolutionNetwork network = {0: '1.0 H2O_g = 1.0 H2O_d', 1: '1.0 CO2_g = 1.0 CO2_d'}


[07:41:50 - atmodeller.reactions           - INFO     ] - ReactionSystem network = {0: '1.0 H2_g + 0.5 O2_g = 1.0 H2O_g',
 1: '0.5 O2_g + 1.0 CO_g = 1.0 CO2_g',
 2: '2.0 H2_g + 1.0 CO_g = 0.5 O2_g + 1.0 CH4_g',
 3: '1.0 H2O_g = 1.0 H2O_d',
 4: '1.0 CO2_g = 1.0 CO2_d'}


Define system constraints, which can be a combination of activity constraints (fugacity constraints for gas species) and mass constraints. For early planetary evolution, it is typical to impose an oxygen fugacity using the Iron-Wüstite buffer, regulated by Fe-FeO equilibrium. Importantly, the number of constraints must equal the number of components (usually the number of elements), less the number of condensates imposed as stable. See the Constraints section of the Species page for details.

In [5]:
# Fugacity constraint for fO2 is set by the Iron-Wuestite buffer
activity_constraints = {"O2_g": IronWustiteBuffer()}

# Mass constraints based on the mass of Earth's oceans and a C/H ratio of 1
oceans = 1
ch_ratio = 1
h_kg = earth.oceans_to_hydrogen_mass(oceans)
c_kg = ch_ratio * h_kg
mass_constraints = {"C": c_kg, "H": h_kg}

Create the model.

In [6]:
parameters = Parameters(
    planet, activity_constraints=activity_constraints, mass_constraints=mass_constraints
)

model = EquilibriumModel(parameters)

Solve the model.

In [7]:
output = model.solve_with_default()

[07:41:50 - atmodeller.classes             - INFO     ] - Initial guess: 1 model(s) from ElementPotentialInitializer, 0 model(s) from the given initial guess


Check that the model was solved.

In [8]:
output.solver_stats_to_logger()

[07:41:54 - atmodeller.output              - INFO     ] - Solve complete: 1 (100.00%) successful model(s)


[07:41:54 - atmodeller.output              - INFO     ] - Solve attempt 1: 1 (100.00%) model(s) solved


[07:41:54 - atmodeller.output              - INFO     ] - Solver steps (max) = 1


## Output

*Atmodeller* provides several ways to access the output. For use in your workflows it is likely most convenient to get a nested dictionary of output quantities.

In [9]:
# By default the returned arrays are JAX arrays, but you can convert any arrays to numpy
# as follows, which is probably your preferred format for further analysis and plotting.
output_dict_default = output.to_dict(to_numpy=True)

print("Dictionary keys for 'named_arrays' output format:", list(output_dict_default.keys()))

# E.g., access the elemental masses in the gas phase
print("Gas phase elemental masses:", output_dict_default["gas"]["elements"]["mass"])

Dictionary keys for 'named_arrays' output format: ['constraints', 'gas', 'initial_guess', 'residual', 'silicate_melt', 'solution', 'solver', 'state']
Gas phase elemental masses: {'C': array([[1.48019714e+20]]), 'H': array([[1.72818947e+17]]), 'O': array([[2.34120923e+20]])}


You can also obtain the output in a different format, for example by elements and species.

In [10]:
output_dict_elements_species = output.to_dict(output_format="elements_species", to_numpy=False)

print(
    "Dictionary keys for 'elements_species' output format:",
    list(output_dict_elements_species.keys()),
)

# E.g., access quantities associated with hydrogen
print("Hydrogen quantities:", output_dict_elements_species["H"])

Dictionary keys for 'elements_species' output format: ['C', 'H', 'O', 'H2O_g', 'H2_g', 'O2_g', 'CO_g', 'CO2_g', 'CH4_g', 'gas', 'H2O_d', 'CO2_d', 'silicate_melt', 'solution', 'initial_guess', 'solver', 'residual', 'state']
Hydrogen quantities: {'gas': {'mass': Array([[1.72818947e+17]], dtype=float64), 'number_moles': Array([[1.71457404e+20]], dtype=float64), 'log_epsilon': Array([[12.]], dtype=float64), 'X_H': Array([[0.]], dtype=float64)}, 'silicate_melt': {'mass': Array([[1.54827335e+20]], dtype=float64), 'number_moles': Array([[1.53607537e+23]], dtype=float64)}, 'constraints': {'mass': Array([[1.55000154e+20]], dtype=float64), 'number_moles': Array([[1.53778995e+23]], dtype=float64)}}


Another useful output option is a dictionary of dataframes, in which the same ``output_format`` options can be specified. The dataframes can also be written to files: set ``WRITE_OUTPUT`` to ``True`` to write them to ``OUTPUT_DIRECTORY``.

In [11]:
# There are a few different output formats
output_format: Literal["elements_species", "named_arrays"] = "named_arrays"

# Whether to export the data to Excel and pickle files, and where
WRITE_OUTPUT = False
OUTPUT_DIRECTORY = Path("output") / "basic_usage"

if WRITE_OUTPUT:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

dataframe_dict = output.to_dataframes(output_format=output_format)

The dataframes can be exported to a pickle file

In [12]:
if WRITE_OUTPUT:
    # Write the data to a pickle file with dataframes
    output.to_pickle(OUTPUT_DIRECTORY / "basic_usage", output_format=output_format)

or to Excel.

In [13]:
if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output.to_excel(OUTPUT_DIRECTORY / "basic_usage", output_format=output_format)

## Batching

*Atmodeller* supports batching: you can provide arrays of input quantities and *Atmodeller* will automatically solve the system for each set of inputs using broadcasting rules. This enables efficient computation of multiple scenarios in a single call.

For example, let's define a new planet and batch over the temperature.

In [14]:
temperature_batch = np.array([1500, 2000, 2500])

planet = Planet.from_species(
    gas_species,
    silicate_melt_species=melt_species,
    temperature=temperature_batch,
    mantle_melt_fraction=1.0,
    planet_mass=earth.mass,
    core_mass_fraction=earth.core_mass_fraction,
    surface_radius=earth.radius,
)

parameters = Parameters(
    planet, activity_constraints=activity_constraints, mass_constraints=mass_constraints
)

model = EquilibriumModel(parameters)

output = model.solve_with_default()

output.solver_stats_to_logger()

[07:41:57 - atmodeller.phases              - INFO     ] - Creating GasPhase: ('H2O_g: IdealGas', 'H2_g: IdealGas', 'O2_g: IdealGas', 'CO_g: IdealGas', 'CO2_g: IdealGas', 'CH4_g: IdealGas')


[07:41:57 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ('H2O_d: SolubilityPowerLaw', 'CO2_d: _CO2_basalt_dixon95')


[07:41:57 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[07:41:57 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[07:41:57 - atmodeller.reactions           - INFO     ] - ReactionNetwork network = {0: '1.0 H2_g + 0.5 O2_g = 1.0 H2O_g',
 1: '0.5 O2_g + 1.0 CO_g = 1.0 CO2_g',
 2: '2.0 H2_g + 1.0 CO_g = 0.5 O2_g + 1.0 CH4_g'}


[07:41:57 - atmodeller.reactions           - INFO     ] - Thermodynamic data of all species cover 200 K to 6000 K, outside which each species follows its extrapolation


... (2 outputs left out)

[07:41:57 - atmodeller.classes             - INFO     ] - Initial guess: 3 model(s) from ElementPotentialInitializer, 0 model(s) from the given initial guess


[07:42:01 - atmodeller.output              - INFO     ] - Solve complete: 3 (100.00%) successful model(s)


[07:42:01 - atmodeller.output              - INFO     ] - Solve attempt 1: 3 (100.00%) model(s) solved


[07:42:01 - atmodeller.output              - INFO     ] - Solver steps (max) = 1


The output will similarly reflect the fact that temperature has been batched. You can also force all the arrays to match the batch size by setting ``expand_to_batch=True``.

In [15]:
output_dict = output.to_dict(to_numpy=True, expand_to_batch=True)

# Let's check that we recover the imposed temperature batch
print("temperature:", output_dict["state"]["temperature"])

# Because we set expand_to_batch=True, even non-batched quantities are returned as arrays
print("radius:", output_dict["state"]["surface_radius"])

temperature: [1500. 2000. 2500.]
radius: [6371000. 6371000. 6371000.]


Finally, you can batch multiple parameters at the same time, as long as they have the same batch size. For example, you can batch both temperature and planetary radius, and they will be paired together for each calculation.

In [16]:
temperature_batch = np.array([1500, 2000, 2500])
radius_batch = np.array([earth.radius, earth.radius * 1.5, earth.radius * 2])

planet = Planet.from_species(
    gas_species,
    silicate_melt_species=melt_species,
    temperature=temperature_batch,
    mantle_melt_fraction=1.0,
    planet_mass=earth.mass,
    core_mass_fraction=earth.core_mass_fraction,
    surface_radius=radius_batch,
)

parameters = Parameters(
    planet, activity_constraints=activity_constraints, mass_constraints=mass_constraints
)

model = EquilibriumModel(parameters)

output = model.solve_with_default()

output.solver_stats_to_logger()

output_dict = output.to_dict(to_numpy=True, expand_to_batch=True)

# Let's check that we recover the imposed temperature batch
print("temperature:", output_dict["state"]["temperature"])

# Because we set expand_to_batch=True, even non-batched quantities are returned as arrays
print("radius:", output_dict["state"]["surface_radius"])

[07:42:03 - atmodeller.phases              - INFO     ] - Creating GasPhase: ('H2O_g: IdealGas', 'H2_g: IdealGas', 'O2_g: IdealGas', 'CO_g: IdealGas', 'CO2_g: IdealGas', 'CH4_g: IdealGas')


[07:42:03 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ('H2O_d: SolubilityPowerLaw', 'CO2_d: _CO2_basalt_dixon95')


[07:42:03 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[07:42:03 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[07:42:03 - atmodeller.reactions           - INFO     ] - ReactionNetwork network = {0: '1.0 H2_g + 0.5 O2_g = 1.0 H2O_g',
 1: '0.5 O2_g + 1.0 CO_g = 1.0 CO2_g',
 2: '2.0 H2_g + 1.0 CO_g = 0.5 O2_g + 1.0 CH4_g'}


[07:42:03 - atmodeller.reactions           - INFO     ] - Thermodynamic data of all species cover 200 K to 6000 K, outside which each species follows its extrapolation


... (3 outputs left out)

[07:42:06 - atmodeller.output              - INFO     ] - Solve complete: 3 (100.00%) successful model(s)


[07:42:06 - atmodeller.output              - INFO     ] - Solve attempt 1: 3 (100.00%) model(s) solved


[07:42:06 - atmodeller.output              - INFO     ] - Solver steps (max) = 1


temperature: [1500. 2000. 2500.]
radius: [ 6371000.  9556500. 12742000.]


## Condensates

Condensates, such as graphite or liquid water, are pure phases selected from the thermodynamic data by formula or by end-member, the gas, liquid or mineral to which the data refer. `PurePhase.from_formula` stitches together the condensed end-members of a formula given by `stitchable_end_members` in order of temperature, and `PurePhase.from_end_member` selects one. A condensate is named from its Hill formula and a suffix, e.g. `C_graphite` or `O2Si_cd`, and this name is its key in the output. See the Species page of the documentation for more.

For example, let's include graphite and batch over temperature at an oxygen fugacity 2 log units below the Iron-Wüstite buffer. A condensate that is not stable has an activity below unity and a negligible mass.

In [17]:
print("Condensed end-members of SiO2:", Database.default().stitchable_end_members("SiO2"))
print("All of them stitched together:", PurePhase.from_formula("SiO2").name)

# Graphite, with the volume of Gustafson (1986) so that its Gibbs energy depends on pressure
graphite = PurePhase.from_end_member("graphite")

planet = Planet.from_species(
    gas_species,
    silicate_melt_species=melt_species,
    temperature=temperature_batch,
    mantle_melt_fraction=1.0,
    planet_mass=earth.mass,
    core_mass_fraction=earth.core_mass_fraction,
    surface_radius=earth.radius,
    condensates=(graphite,),
)

parameters = Parameters(
    planet,
    activity_constraints={"O2_g": IronWustiteBuffer(-2)},
    mass_constraints=mass_constraints,
)

model = EquilibriumModel(parameters)

output = model.solve_with_default()

output.solver_stats_to_logger()

graphite_output = output.to_dict(to_numpy=True)[graphite.name]["species"]
print(graphite.name, "activity:", graphite_output["activity"][graphite.name].ravel())
print(graphite.name, "mass (kg):", graphite_output["mass"][graphite.name].ravel())

[07:42:06 - atmodeller.phases              - INFO     ] - Creating PurePhase: ('O2Si_cd: ActivityCoefficient',)


[07:42:06 - atmodeller.phases              - INFO     ] - Creating PurePhase: ('C_graphite: ActivityCoefficient',)


[07:42:06 - atmodeller.phases              - INFO     ] - Creating GasPhase: ('H2O_g: IdealGas', 'H2_g: IdealGas', 'O2_g: IdealGas', 'CO_g: IdealGas', 'CO2_g: IdealGas', 'CH4_g: IdealGas')


[07:42:06 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ('H2O_d: SolubilityPowerLaw', 'CO2_d: _CO2_basalt_dixon95')


[07:42:06 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


[07:42:06 - atmodeller.phases              - INFO     ] - Creating CondensedPhase: ()


... (6 outputs left out)

[07:42:14 - atmodeller.output              - INFO     ] - Solve complete: 3 (100.00%) successful model(s)


[07:42:14 - atmodeller.output              - INFO     ] - Solve attempt 1: 3 (100.00%) model(s) solved


[07:42:14 - atmodeller.output              - INFO     ] - Solver steps (max) = 1


C_graphite activity: [1.         0.07521102 0.01426341]
C_graphite mass (kg): [3.99029959e+19 5.99044236e-06 3.64701263e-06]
